## Preprocessing

In [ ]:
import os
import re
import numpy as np
import pandas as pd
import scanpy as sc
import scipy.sparse as sp
import harmonypy as hm
import matplotlib as mpl
import matplotlib.pyplot as plt
from pathlib import Path

# hard-set based on your terminal output
os.environ["PATH"] = "/group/canc2/anson/miniconda3/envs/python310/bin:" + os.environ.get("PATH", "")
os.environ["R_HOME"] = "/group/canc2/anson/miniconda3/envs/python310/lib/R"

# Optional Harmony
HAVE_HARMONY = True

# Plot defaults
sc.settings.verbosity = 3
sc.settings.set_figure_params(dpi=1200, facecolor='white', fontsize=8)
mpl.rcParams['savefig.dpi'] = 1200
mpl.rcParams['savefig.bbox'] = 'tight'


In [ ]:
# --------------------
# Parameters
# --------------------
ROOT = "/group/canc2/anson/working/cf-pbmc-bal"
PBMC_H5AD = os.path.join(ROOT, "data", "SCEs", "analysis", "pbmc.h5ad")
BAL_H5AD  = os.path.join(ROOT, "data", "SCEs", "analysis", "bal.h5ad")

CONDITION_KEY = "Condition2"
BATCH_KEY = "sampleID"
CLUSTER_KEY = "ann_lv2"

COUNTS_LAYER = "counts"

N_HVG = 2000
N_PCS = 20
N_NEIGHBORS = 15

OUTDIR = "paga_pbmc_bal_by_condition2"
os.makedirs(OUTDIR, exist_ok=True)
sc.settings.figdir = OUTDIR

# Match your Condition2 palette
cond_colors = {"C": "#26547c", "CFN": "#ffd166", "CFB": "#ef476f"}

## Helpers: integer counts layer

In [ ]:
def fraction_non_integer_like(X, tol=1e-6, sample_n=200000, seed=0):
    if sp.issparse(X):
        vals = X.data
    else:
        vals = np.asarray(X).ravel()
    if vals.size == 0:
        return 0.0
    rng = np.random.default_rng(seed)
    if vals.size > sample_n:
        vals = vals[rng.choice(vals.size, size=sample_n, replace=False)]
    return float(np.mean(np.abs(vals - np.round(vals)) > tol))


def ensure_counts_layer_from_X(adata: sc.AnnData, layer_key: str = COUNTS_LAYER, tol=1e-6, max_frac_nonint=1e-4):
    if layer_key in adata.layers:
        return
    frac = fraction_non_integer_like(adata.X, tol=tol)
    print(f"fraction non-integer-like in adata.X: {frac:.6g}")
    if frac > max_frac_nonint:
        raise ValueError(
            f"adata.X is not integer-like enough to treat as raw counts (frac_nonint={frac}).\n"
            "Re-convert/export with counts preserved is recommended."
        )
    if sp.issparse(adata.X):
        Xc = adata.X.copy()
        Xc.data = np.rint(Xc.data).astype(np.int32)
        adata.layers[layer_key] = Xc
    else:
        adata.layers[layer_key] = np.rint(np.asarray(adata.X)).astype(np.int32)


def set_counts_as_X(adata: sc.AnnData, layer_key: str = COUNTS_LAYER):
    if layer_key not in adata.layers:
        raise KeyError(f"Missing layer '{layer_key}'")
    adata.X = adata.layers[layer_key].copy()
    return adata


## Load PBMC + BAL

In [ ]:
adata_pbmc = sc.read_h5ad(PBMC_H5AD)
adata_bal = sc.read_h5ad(BAL_H5AD)

for name, ad in [("pbmc", adata_pbmc), ("bal", adata_bal)]:
    for k in [CONDITION_KEY, BATCH_KEY, CLUSTER_KEY]:
        if k not in ad.obs.columns:
            raise KeyError(f"[{name}] Expected obs column '{k}' not found.")
    print(f"{name}: shape={ad.shape} layers={list(ad.layers.keys())}")
    print(f"{name}: X min/max={float(ad.X.min())} {float(ad.X.max())}")

print("Ensuring counts layers...")
ensure_counts_layer_from_X(adata_pbmc, layer_key=COUNTS_LAYER)
ensure_counts_layer_from_X(adata_bal, layer_key=COUNTS_LAYER)
print("pbmc layers:", list(adata_pbmc.layers.keys()))
print("bal layers:", list(adata_bal.layers.keys()))


In [ ]:
# -------------------------
# Normalize and Log-Transform
# -------------------------
for dataset in [adata_pbmc, adata_bal]:
    sc.pp.normalize_total(dataset, target_sum=1e4)
    sc.pp.log1p(dataset)


## Visualization

### Dotplot

In [ ]:
import scanpy as sc

# -----------------------
# 1) Subset BAL to CD4T
# -----------------------
ct_key_lv2 = "ann_lv1"

adata_cdc = adata_bal[
    adata_bal.obs[ct_key_lv2].astype(str).isin(["CD4T"])
].copy()

# drop unused categories (Seurat fct_drop equivalent)
for k in ["ann_lv1", "ann_lv2", "Condition2"]:
    if k in adata_cdc.obs.columns and hasattr(adata_cdc.obs[k], "cat"):
        adata_cdc.obs[k] = adata_cdc.obs[k].cat.remove_unused_categories()

# -----------------------
# 2) Gene list
# -----------------------
selected = [
    "IKZF2","KLF2","ITGAE","ZNF683","RUNX3","XCL2", # TRM
    "RORC","RORA","KLRB1","IL23R","CCR6","DPP4","CXCR3","CXCR6", # Th17
    "CSF2","IL17A","IL17F","IL21","IL22"
]

selected_present = [g for g in selected if g in adata_cdc.var_names]
if len(selected_present) < len(selected):
    print("Missing genes:", set(selected) - set(selected_present))

# -----------------------
# 3) DotPlot grouped by Condition2
# -----------------------
dp = sc.pl.dotplot(
    adata_cdc,
    var_names=selected_present,
    groupby="Condition2",
    standard_scale="var",
    mean_only_expressed=True,
    cmap="Blues",
    dot_max=0.6,
    show=False,   # capture the object / returned dict
)

# Get the figure robustly whether dp is a dict or has .fig
if isinstance(dp, dict):
    fig = dp.get("fig", plt.gcf())
elif hasattr(dp, "fig"):
    fig = dp.fig
else:
    fig = plt.gcf()

# Find the main plot axes (heuristic: an axes without a colorbar and with some ticks/labels)
main_ax = None
for ax in fig.axes:
    if getattr(ax, "colorbar", None) is None:
        # prefer axes that have x or y ticklabels (to avoid legend-only axes)
        if (len(ax.get_xticklabels()) > 0) or (len(ax.get_yticklabels()) > 0):
            main_ax = ax
            break
# fallback
if main_ax is None:
    main_ax = fig.axes[0]

# Set the title on the main axis so it sits closer to the plot; adjust `pad` to taste
main_ax.set_title("BAL CD4T", fontsize=10, pad=24)  # try pad=6,8,10 ... smaller -> closer
main_ax.set_xlabel(main_ax.get_xlabel(), fontsize=8)
main_ax.set_ylabel(main_ax.get_ylabel(), fontsize=8)

plt.setp(main_ax.get_xticklabels(), fontstyle="italic")

# Keep only min/max labels on any colorbars present
for ax in fig.axes:
    cb = getattr(ax, "colorbar", None)
    if cb is not None:
        vmin, vmax = cb.vmin, cb.vmax
        cb.set_ticks([vmin, vmax])
        cb.set_ticklabels([f"{vmin:.2f}", f"{vmax:.2f}"])

# Tighten layout (you can tune rect/top if needed)
fig.tight_layout()
fig.show()

# Save figure
outdir = Path("/group/canc2/anson/working/cf-pbmc-bal/data/plots/BAL_CD4T")
outdir.mkdir(parents=True, exist_ok=True)
out = os.path.join(outdir, f"BAL-CD4T.DEGdotplot.jpeg")
fig.savefig(out, dpi=1600, format="jpeg", bbox_inches="tight", facecolor="white")


In [ ]:
import scanpy as sc

# -----------------------
# 1) Subset PBMC to CD8T.naive
# -----------------------
ct_key_lv2 = "ann_lv2"

adata_cdc = adata_pbmc[
    adata_pbmc.obs[ct_key_lv2].astype(str).isin(["CD4T.naive"])
].copy()

# drop unused categories (Seurat fct_drop equivalent)
for k in ["ann_lv1", "ann_lv2", "Condition2"]:
    if k in adata_cdc.obs.columns and hasattr(adata_cdc.obs[k], "cat"):
        adata_cdc.obs[k] = adata_cdc.obs[k].cat.remove_unused_categories()

# -----------------------
# 2) Gene list
# -----------------------
selected = [
    "JUN","FOS","JUNB","JUND","DUSP1","CSRNP1","NR4A1"
]

selected_present = [g for g in selected if g in adata_cdc.var_names]
if len(selected_present) < len(selected):
    print("Missing genes:", set(selected) - set(selected_present))

# -----------------------
# 3) DotPlot grouped by Condition2
# -----------------------
dp = sc.pl.dotplot(
    adata_cdc,
    var_names=selected_present,
    groupby="Condition2",
    standard_scale="var",
    mean_only_expressed=True,
    dot_max=0.6,
    show=False,   # capture the object / returned dict
)

# Get the figure robustly whether dp is a dict or has .fig
if isinstance(dp, dict):
    fig = dp.get("fig", plt.gcf())
elif hasattr(dp, "fig"):
    fig = dp.fig
else:
    fig = plt.gcf()

# Find the main plot axes (heuristic: an axes without a colorbar and with some ticks/labels)
main_ax = None
for ax in fig.axes:
    if getattr(ax, "colorbar", None) is None:
        # prefer axes that have x or y ticklabels (to avoid legend-only axes)
        if (len(ax.get_xticklabels()) > 0) or (len(ax.get_yticklabels()) > 0):
            main_ax = ax
            break
# fallback
if main_ax is None:
    main_ax = fig.axes[0]

# Set the title on the main axis so it sits closer to the plot; adjust `pad` to taste
main_ax.set_title("PBMC CD4T.naive", fontsize=10, pad=24)  # try pad=6,8,10 ... smaller -> closer
plt.setp(main_ax.get_xticklabels(), fontstyle="italic")

# Keep only min/max labels on any colorbars present
for ax in fig.axes:
    cb = getattr(ax, "colorbar", None)
    if cb is not None:
        vmin, vmax = cb.vmin, cb.vmax
        cb.set_ticks([vmin, vmax])
        cb.set_ticklabels([f"{vmin:.2f}", f"{vmax:.2f}"])

# Tighten layout (you can tune rect/top if needed)
fig.tight_layout()
fig.show()

# Save figure
outdir = Path("/group/canc2/anson/working/cf-pbmc-bal/data/plots/PBMC_CD4T.naive")
outdir.mkdir(parents=True, exist_ok=True)
out = os.path.join(outdir, f"PBMC-CD4T.naive_trmDEGdotplot.jpeg")
fig.savefig(out, dpi=1600, format="jpeg", bbox_inches="tight", facecolor="white")


In [ ]:
import scanpy as sc

# -----------------------
# 1) Subset PBMC to CD8T.naive
# -----------------------
ct_key_lv2 = "ann_lv2"

adata_cdc = adata_pbmc[
    adata_pbmc.obs[ct_key_lv2].astype(str).isin(["CD4T.naive"])
].copy()

# drop unused categories (Seurat fct_drop equivalent)
for k in ["ann_lv1", "ann_lv2", "Condition2"]:
    if k in adata_cdc.obs.columns and hasattr(adata_cdc.obs[k], "cat"):
        adata_cdc.obs[k] = adata_cdc.obs[k].cat.remove_unused_categories()

# -----------------------
# 2) Gene list
# -----------------------
selected = [
    "JUN","FOS","JUNB","JUND","DUSP1","CSRNP1","NR4A1"
]

selected_present = [g for g in selected if g in adata_cdc.var_names]
if len(selected_present) < len(selected):
    print("Missing genes:", set(selected) - set(selected_present))

# -----------------------
# 3) DotPlot grouped by Condition2
# -----------------------
dp = sc.pl.dotplot(
    adata_cdc,
    var_names=selected_present,
    groupby="Condition2",
    standard_scale="var",
    mean_only_expressed=True,
    dot_max=0.6,
    show=False,   # capture the object / returned dict
)

# Get the figure robustly whether dp is a dict or has .fig
if isinstance(dp, dict):
    fig = dp.get("fig", plt.gcf())
elif hasattr(dp, "fig"):
    fig = dp.fig
else:
    fig = plt.gcf()

# Find the main plot axes (heuristic: an axes without a colorbar and with some ticks/labels)
main_ax = None
for ax in fig.axes:
    if getattr(ax, "colorbar", None) is None:
        # prefer axes that have x or y ticklabels (to avoid legend-only axes)
        if (len(ax.get_xticklabels()) > 0) or (len(ax.get_yticklabels()) > 0):
            main_ax = ax
            break
# fallback
if main_ax is None:
    main_ax = fig.axes[0]

# Set the title on the main axis so it sits closer to the plot; adjust `pad` to taste
main_ax.set_title("PBMC CD4T.naive", fontsize=10, pad=24)  # try pad=6,8,10 ... smaller -> closer
plt.setp(main_ax.get_xticklabels(), fontstyle="italic")

# Keep only min/max labels on any colorbars present
for ax in fig.axes:
    cb = getattr(ax, "colorbar", None)
    if cb is not None:
        vmin, vmax = cb.vmin, cb.vmax
        cb.set_ticks([vmin, vmax])
        cb.set_ticklabels([f"{vmin:.2f}", f"{vmax:.2f}"])

# Tighten layout (you can tune rect/top if needed)
fig.tight_layout()
fig.show()

# Save figure
outdir = Path("/group/canc2/anson/working/cf-pbmc-bal/data/plots/PBMC_CD4T.naive")
outdir.mkdir(parents=True, exist_ok=True)
out = os.path.join(outdir, f"PBMC-CD4T.naive_trmDEGdotplot.jpeg")
fig.savefig(out, dpi=1600, format="jpeg", bbox_inches="tight", facecolor="white")


## Kruskal–Wallis test of Th17

### %RORC+

In [ ]:
import numpy as np
import pandas as pd
import scipy.sparse as sp
from scipy.stats import kruskal
import scikit_posthocs as sp_post

# -----------------------
# Settings (BAL only)
# -----------------------
CD4_KEY = "ann_lv1"
CD4_LABEL = "CD4T"
GROUP_KEY = "Condition2"

# internal donor/sample id key (BAL)
ID_KEY = "donor_id" if "donor_id" in adata_bal.obs.columns else "sampleID"

gene_sets = {
    # AND across genes (RORC+ AND CCR6+)
    "IL17_like": ["RORC"],
}

USE_LAYER = "counts"   # set to None to use adata.X
POS_THRESH = 0.0

MIN_CELLS_PER_DONOR = 5
ORDER = ["C", "CFN", "CFB"]

# -----------------------
# Helpers
# -----------------------
def _to_1d_dense(x):
    if sp.issparse(x):
        x = x.toarray()
    return np.asarray(x).ravel()

def _get_matrix(adata, layer=None):
    if layer is None:
        return adata.X
    if layer not in adata.layers:
        raise KeyError(f"Layer {layer!r} not found. Available layers: {list(adata.layers.keys())}")
    return adata.layers[layer]

def gene_set_positive_AND(adata, genes, thresh=0.0, layer=None):
    """
    Cell-level positivity for a geneset using AND:
      positive if ALL genes have expression > thresh.
    Returns: (pos_bool_vector, present_genes)
    """
    present = [g for g in genes if g in adata.var_names]
    if len(present) == 0:
        return None, []

    X = _get_matrix(adata[:, present], layer=layer)

    masks = []
    for j in range(len(present)):
        col = _to_1d_dense(X[:, j])
        masks.append(col > thresh)

    pos = np.logical_and.reduce(masks)
    return pos, present

def donor_percent_table(adata, id_key, group_key, gene_sets, thresh=0.0, min_cells=0, layer=None):
    """
    Donor-level summary:
      donor_id, Condition2, n_cells, n_pos_<gs>, pct_<gs>

    Note:
      - This function assumes `adata` is already subset to the population of interest (e.g. CD4T).
      - It reports donor-level percent positive among those cells.
    """
    if id_key not in adata.obs.columns:
        raise KeyError(f"Missing obs column {id_key!r}")
    if group_key not in adata.obs.columns:
        raise KeyError(f"Missing obs column {group_key!r}")

    obs = adata.obs[[id_key, group_key]].copy()
    obs[id_key] = obs[id_key].astype("string").str.strip()
    obs[group_key] = obs[group_key].astype("string").str.strip()
    obs = obs.dropna(subset=[id_key, group_key])

    base = (
        obs.groupby([id_key, group_key], observed=True)
           .size()
           .rename("n_cells")
           .reset_index()
    )

    for gs_name, genes in gene_sets.items():
        pos, present = gene_set_positive_AND(adata, genes, thresh=thresh, layer=layer)
        if pos is None:
            print(f"[{gs_name}] WARNING: none of genes present: {genes}")
            base[f"n_pos_{gs_name}"] = 0
            base[f"pct_{gs_name}"] = 0.0
            continue

        print(f"[{gs_name}] genes present ({len(present)}): {present}  (AND positivity)")

        tmp = obs[[id_key, group_key]].copy()
        tmp["pos"] = pos.astype(np.int32)

        tmp = (
            tmp.groupby([id_key, group_key], observed=True)["pos"]
               .sum()
               .rename(f"n_pos_{gs_name}")
               .reset_index()
        )

        base = base.merge(tmp, on=[id_key, group_key], how="left")
        base[f"n_pos_{gs_name}"] = base[f"n_pos_{gs_name}"].fillna(0).astype(int)
        base[f"pct_{gs_name}"] = base[f"n_pos_{gs_name}"] / base["n_cells"] * 100.0

    if min_cells and min_cells > 0:
        base = base.loc[base["n_cells"] >= min_cells].copy()

    return base.rename(columns={id_key: "donor_id", group_key: "Condition2"})

# -----------------------
# 0) Prepare dataset (BAL only)
# -----------------------
adata_cd4 = adata_bal[adata_bal.obs[CD4_KEY].astype(str) == CD4_LABEL].copy()

if USE_LAYER is not None and USE_LAYER not in adata_cd4.layers:
    raise KeyError(
        f"adata_cd4 is missing layers[{USE_LAYER!r}]. Available: {list(adata_cd4.layers.keys())}"
    )

# -----------------------
# 1) Donor-level table (BAL only)
# -----------------------
df = donor_percent_table(
    adata_cd4,
    id_key=ID_KEY,
    group_key=GROUP_KEY,
    gene_sets=gene_sets,
    thresh=POS_THRESH,
    min_cells=MIN_CELLS_PER_DONOR,
    layer=USE_LAYER,
)

# key used by your plotting code for n per group
df["donor_key"] = df["donor_id"].astype(str)

# enforce ordering & filter to requested conditions
df = df.loc[df["Condition2"].astype(str).isin(ORDER)].copy()
df["Condition2"] = pd.Categorical(df["Condition2"].astype(str), categories=ORDER, ordered=True)

print("\nUnique donors per Condition2 (BAL only):")
print(df.groupby("Condition2", observed=True)["donor_key"].nunique().to_string())

# -----------------------
# 2) Kruskal–Wallis + Dunn posthoc (Bonferroni-adjusted within Dunn)
# -----------------------
group_col = "Condition2"
metric = "pct_IL17_like"
order = ORDER

groups_used = [
    g for g in order
    if df.loc[df[group_col].astype(str) == g, metric].dropna().shape[0] > 0
]

# Omnibus Kruskal–Wallis
vals = [df.loc[df[group_col].astype(str) == g, metric].dropna().to_numpy() for g in groups_used]
H, p_kw = (kruskal(*vals) if len(vals) >= 2 else (np.nan, np.nan))

kw_summary = pd.DataFrame([{
    "test": "Kruskal–Wallis (omnibus)",
    "groups_used": ", ".join(groups_used),
    "H": float(H) if np.isfinite(H) else np.nan,
    "p": float(p_kw) if np.isfinite(p_kw) else np.nan,
}])

# Dunn post-hoc (Bonferroni adjustment handled inside posthoc_dunn)
posthoc_df = pd.DataFrame(columns=["comparison", "n_a", "n_b", "median_a", "median_b", "p_adj"])

if len(groups_used) >= 2:
    df_ph = df.loc[df[group_col].astype(str).isin(groups_used), [group_col, metric]].dropna().copy()
    df_ph[group_col] = df_ph[group_col].astype(str)

    pmat = sp_post.posthoc_dunn(
        df_ph,
        val_col=metric,
        group_col=group_col,
        p_adjust="bonferroni",
    )

    # only compute pairs among groups_used, but keep ORDER-based ordering
    pairs = [(order[i], order[j]) for i in range(len(order)) for j in range(i + 1, len(order))]
    pairs = [(a, b) for (a, b) in pairs if (a in groups_used and b in groups_used)]

    rows = []
    for a, b in pairs:
        x = df_ph.loc[df_ph[group_col] == a, metric].to_numpy()
        y = df_ph.loc[df_ph[group_col] == b, metric].to_numpy()
        rows.append({
            "comparison": f"{a} vs {b}",
            "n_a": int(len(x)),
            "n_b": int(len(y)),
            "median_a": float(np.median(x)) if len(x) else np.nan,
            "median_b": float(np.median(y)) if len(y) else np.nan,
            "p_adj": float(pmat.loc[a, b]),  # Bonferroni-adjusted p-value
        })

    posthoc_df = pd.DataFrame(rows)
    posthoc_df["comparison"] = pd.Categorical(
        posthoc_df["comparison"],
        categories=[f"{a} vs {b}" for a, b in pairs],
        ordered=True,
    )
    posthoc_df = posthoc_df.sort_values("comparison")

# display/print (same objects your plot code expects: df, metric, posthoc_df, p_kw)
try:
    display(kw_summary)
    display(posthoc_df)
except NameError:
    print(kw_summary.to_string(index=False))
    if len(posthoc_df):
        print(posthoc_df.to_string(index=False))
    else:
        print("(posthoc_df is empty: need >=2 groups with data)")

### %IL17+

In [ ]:
import numpy as np
import pandas as pd
import scipy.sparse as sp
from scipy.stats import kruskal
import scikit_posthocs as sp_post

# -----------------------
# Settings (BAL only)
# -----------------------
CD4_KEY = "ann_lv1"
CD4_LABEL = "CD4T"
GROUP_KEY = "Condition2"

# internal donor/sample id key (BAL)
ID_KEY = "donor_id" if "donor_id" in adata_bal.obs.columns else "sampleID"

gene_sets = {
    # OR across genes (IL17A OR IL17F)
    "IL17_like": ["IL17A", "IL17F"],
}

USE_LAYER = "counts"   # set to None to use adata.X
POS_THRESH = 0.0

MIN_CELLS_PER_DONOR = 5
ORDER = ["C", "CFN", "CFB"]

# -----------------------
# Helpers
# -----------------------
def _to_1d_dense(x):
    if sp.issparse(x):
        x = x.toarray()
    return np.asarray(x).ravel()

def _get_matrix(adata, layer=None):
    if layer is None:
        return adata.X
    if layer not in adata.layers:
        raise KeyError(f"Layer {layer!r} not found. Available layers: {list(adata.layers.keys())}")
    return adata.layers[layer]

def gene_set_positive_OR(adata, genes, thresh=0.0, layer=None):
    """
    Cell-level positivity for a geneset using OR:
      positive if ANY gene has expression > thresh.
    Returns: (pos_bool_vector, present_genes)
    """
    present = [g for g in genes if g in adata.var_names]
    if len(present) == 0:
        return None, []

    X = _get_matrix(adata[:, present], layer=layer)

    masks = []
    for j in range(len(present)):
        col = _to_1d_dense(X[:, j])
        masks.append(col > thresh)

    pos = np.logical_or.reduce(masks)
    return pos, present

def donor_percent_table(adata, id_key, group_key, gene_sets, thresh=0.0, min_cells=0, layer=None):
    """
    Donor-level summary:
      donor_id, Condition2, n_cells, n_pos_<gs>, pct_<gs>

    Note:
      - This function assumes `adata` is already subset to the population of interest (e.g. CD4T).
      - It reports donor-level percent positive among those cells.
    """
    if id_key not in adata.obs.columns:
        raise KeyError(f"Missing obs column {id_key!r}")
    if group_key not in adata.obs.columns:
        raise KeyError(f"Missing obs column {group_key!r}")

    obs = adata.obs[[id_key, group_key]].copy()
    obs[id_key] = obs[id_key].astype("string").str.strip()
    obs[group_key] = obs[group_key].astype("string").str.strip()
    obs = obs.dropna(subset=[id_key, group_key])

    base = (
        obs.groupby([id_key, group_key], observed=True)
           .size()
           .rename("n_cells")
           .reset_index()
    )

    for gs_name, genes in gene_sets.items():
        pos, present = gene_set_positive_OR(adata, genes, thresh=thresh, layer=layer)
        if pos is None:
            print(f"[{gs_name}] WARNING: none of genes present: {genes}")
            base[f"n_pos_{gs_name}"] = 0
            base[f"pct_{gs_name}"] = 0.0
            continue

        print(f"[{gs_name}] genes present ({len(present)}): {present}  (OR positivity)")

        tmp = obs[[id_key, group_key]].copy()
        tmp["pos"] = pos.astype(np.int32)

        tmp = (
            tmp.groupby([id_key, group_key], observed=True)["pos"]
               .sum()
               .rename(f"n_pos_{gs_name}")
               .reset_index()
        )

        base = base.merge(tmp, on=[id_key, group_key], how="left")
        base[f"n_pos_{gs_name}"] = base[f"n_pos_{gs_name}"].fillna(0).astype(int)
        base[f"pct_{gs_name}"] = base[f"n_pos_{gs_name}"] / base["n_cells"] * 100.0

    if min_cells and min_cells > 0:
        base = base.loc[base["n_cells"] >= min_cells].copy()

    return base.rename(columns={id_key: "donor_id", group_key: "Condition2"})

# -----------------------
# 0) Prepare dataset (BAL only)
# -----------------------
adata_cd4 = adata_bal[adata_bal.obs[CD4_KEY].astype(str) == CD4_LABEL].copy()

if USE_LAYER is not None and USE_LAYER not in adata_cd4.layers:
    raise KeyError(
        f"adata_cd4 is missing layers[{USE_LAYER!r}]. Available: {list(adata_cd4.layers.keys())}"
    )

# -----------------------
# 1) Donor-level table (BAL only)
# -----------------------
df = donor_percent_table(
    adata_cd4,
    id_key=ID_KEY,
    group_key=GROUP_KEY,
    gene_sets=gene_sets,
    thresh=POS_THRESH,
    min_cells=MIN_CELLS_PER_DONOR,
    layer=USE_LAYER,
)

# key used by your plotting code for n per group
df["donor_key"] = df["donor_id"].astype(str)

# enforce ordering & filter to requested conditions
df = df.loc[df["Condition2"].astype(str).isin(ORDER)].copy()
df["Condition2"] = pd.Categorical(df["Condition2"].astype(str), categories=ORDER, ordered=True)

print("\nUnique donors per Condition2 (BAL only):")
print(df.groupby("Condition2", observed=True)["donor_key"].nunique().to_string())

# -----------------------
# 2) Kruskal–Wallis + Dunn posthoc (Bonferroni-adjusted within Dunn)
# -----------------------
group_col = "Condition2"
metric = "pct_IL17_like"
order = ORDER

groups_used = [
    g for g in order
    if df.loc[df[group_col].astype(str) == g, metric].dropna().shape[0] > 0
]

# Omnibus Kruskal–Wallis
vals = [df.loc[df[group_col].astype(str) == g, metric].dropna().to_numpy() for g in groups_used]
H, p_kw = (kruskal(*vals) if len(vals) >= 2 else (np.nan, np.nan))

kw_summary = pd.DataFrame([{
    "test": "Kruskal–Wallis (omnibus)",
    "groups_used": ", ".join(groups_used),
    "H": float(H) if np.isfinite(H) else np.nan,
    "p": float(p_kw) if np.isfinite(p_kw) else np.nan,
}])

# Dunn post-hoc (Bonferroni adjustment handled inside posthoc_dunn)
posthoc_df = pd.DataFrame(columns=["comparison", "n_a", "n_b", "median_a", "median_b", "p_adj"])

if len(groups_used) >= 2:
    df_ph = df.loc[df[group_col].astype(str).isin(groups_used), [group_col, metric]].dropna().copy()
    df_ph[group_col] = df_ph[group_col].astype(str)

    pmat = sp_post.posthoc_dunn(
        df_ph,
        val_col=metric,
        group_col=group_col,
        p_adjust="bonferroni",
    )

    # only compute pairs among groups_used, but keep ORDER-based ordering
    pairs = [(order[i], order[j]) for i in range(len(order)) for j in range(i + 1, len(order))]
    pairs = [(a, b) for (a, b) in pairs if (a in groups_used and b in groups_used)]

    rows = []
    for a, b in pairs:
        x = df_ph.loc[df_ph[group_col] == a, metric].to_numpy()
        y = df_ph.loc[df_ph[group_col] == b, metric].to_numpy()
        rows.append({
            "comparison": f"{a} vs {b}",
            "n_a": int(len(x)),
            "n_b": int(len(y)),
            "median_a": float(np.median(x)) if len(x) else np.nan,
            "median_b": float(np.median(y)) if len(y) else np.nan,
            "p_adj": float(pmat.loc[a, b]),  # Bonferroni-adjusted p-value
        })

    posthoc_df = pd.DataFrame(rows)
    posthoc_df["comparison"] = pd.Categorical(
        posthoc_df["comparison"],
        categories=[f"{a} vs {b}" for a, b in pairs],
        ordered=True,
    )
    posthoc_df = posthoc_df.sort_values("comparison")

# display/print (same objects your plot code expects: df, metric, posthoc_df, p_kw)
try:
    display(kw_summary)
    display(posthoc_df)
except NameError:
    print(kw_summary.to_string(index=False))
    if len(posthoc_df):
        print(posthoc_df.to_string(index=False))
    else:
        print("(posthoc_df is empty: need >=2 groups with data)")

### %CSF2+

In [ ]:
import numpy as np
import pandas as pd
import scipy.sparse as sp
from scipy.stats import kruskal
import scikit_posthocs as sp_post

# -----------------------
# Settings (BAL only)
# -----------------------
CD4_KEY = "ann_lv1"
CD4_LABEL = "CD4T"
GROUP_KEY = "Condition2"

# internal donor/sample id key (BAL)
ID_KEY = "donor_id" if "donor_id" in adata_bal.obs.columns else "sampleID"

gene_sets = {
    # AND across genes (RORC+ AND CCR6+)
    "IL17_like": ["CSF2","RORC"],
}

USE_LAYER = "counts"   # set to None to use adata.X
POS_THRESH = 0.0

MIN_CELLS_PER_DONOR = 5
ORDER = ["C", "CFN", "CFB"]

# -----------------------
# Helpers
# -----------------------
def _to_1d_dense(x):
    if sp.issparse(x):
        x = x.toarray()
    return np.asarray(x).ravel()

def _get_matrix(adata, layer=None):
    if layer is None:
        return adata.X
    if layer not in adata.layers:
        raise KeyError(f"Layer {layer!r} not found. Available layers: {list(adata.layers.keys())}")
    return adata.layers[layer]

def gene_set_positive_AND(adata, genes, thresh=0.0, layer=None):
    """
    Cell-level positivity for a geneset using AND:
      positive if ALL genes have expression > thresh.
    Returns: (pos_bool_vector, present_genes)
    """
    present = [g for g in genes if g in adata.var_names]
    if len(present) == 0:
        return None, []

    X = _get_matrix(adata[:, present], layer=layer)

    masks = []
    for j in range(len(present)):
        col = _to_1d_dense(X[:, j])
        masks.append(col > thresh)

    pos = np.logical_and.reduce(masks)
    return pos, present

def donor_percent_table(adata, id_key, group_key, gene_sets, thresh=0.0, min_cells=0, layer=None):
    """
    Donor-level summary:
      donor_id, Condition2, n_cells, n_pos_<gs>, pct_<gs>

    Note:
      - This function assumes `adata` is already subset to the population of interest (e.g. CD4T).
      - It reports donor-level percent positive among those cells.
    """
    if id_key not in adata.obs.columns:
        raise KeyError(f"Missing obs column {id_key!r}")
    if group_key not in adata.obs.columns:
        raise KeyError(f"Missing obs column {group_key!r}")

    obs = adata.obs[[id_key, group_key]].copy()
    obs[id_key] = obs[id_key].astype("string").str.strip()
    obs[group_key] = obs[group_key].astype("string").str.strip()
    obs = obs.dropna(subset=[id_key, group_key])

    base = (
        obs.groupby([id_key, group_key], observed=True)
           .size()
           .rename("n_cells")
           .reset_index()
    )

    for gs_name, genes in gene_sets.items():
        pos, present = gene_set_positive_AND(adata, genes, thresh=thresh, layer=layer)
        if pos is None:
            print(f"[{gs_name}] WARNING: none of genes present: {genes}")
            base[f"n_pos_{gs_name}"] = 0
            base[f"pct_{gs_name}"] = 0.0
            continue

        print(f"[{gs_name}] genes present ({len(present)}): {present}  (AND positivity)")

        tmp = obs[[id_key, group_key]].copy()
        tmp["pos"] = pos.astype(np.int32)

        tmp = (
            tmp.groupby([id_key, group_key], observed=True)["pos"]
               .sum()
               .rename(f"n_pos_{gs_name}")
               .reset_index()
        )

        base = base.merge(tmp, on=[id_key, group_key], how="left")
        base[f"n_pos_{gs_name}"] = base[f"n_pos_{gs_name}"].fillna(0).astype(int)
        base[f"pct_{gs_name}"] = base[f"n_pos_{gs_name}"] / base["n_cells"] * 100.0

    if min_cells and min_cells > 0:
        base = base.loc[base["n_cells"] >= min_cells].copy()

    return base.rename(columns={id_key: "donor_id", group_key: "Condition2"})

# -----------------------
# 0) Prepare dataset (BAL only)
# -----------------------
adata_cd4 = adata_bal[adata_bal.obs[CD4_KEY].astype(str) == CD4_LABEL].copy()

if USE_LAYER is not None and USE_LAYER not in adata_cd4.layers:
    raise KeyError(
        f"adata_cd4 is missing layers[{USE_LAYER!r}]. Available: {list(adata_cd4.layers.keys())}"
    )

# -----------------------
# 1) Donor-level table (BAL only)
# -----------------------
df = donor_percent_table(
    adata_cd4,
    id_key=ID_KEY,
    group_key=GROUP_KEY,
    gene_sets=gene_sets,
    thresh=POS_THRESH,
    min_cells=MIN_CELLS_PER_DONOR,
    layer=USE_LAYER,
)

# key used by your plotting code for n per group
df["donor_key"] = df["donor_id"].astype(str)

# enforce ordering & filter to requested conditions
df = df.loc[df["Condition2"].astype(str).isin(ORDER)].copy()
df["Condition2"] = pd.Categorical(df["Condition2"].astype(str), categories=ORDER, ordered=True)

print("\nUnique donors per Condition2 (BAL only):")
print(df.groupby("Condition2", observed=True)["donor_key"].nunique().to_string())

# -----------------------
# 2) Kruskal–Wallis + Dunn posthoc (Bonferroni-adjusted within Dunn)
# -----------------------
group_col = "Condition2"
metric = "pct_IL17_like"
order = ORDER

groups_used = [
    g for g in order
    if df.loc[df[group_col].astype(str) == g, metric].dropna().shape[0] > 0
]

# Omnibus Kruskal–Wallis
vals = [df.loc[df[group_col].astype(str) == g, metric].dropna().to_numpy() for g in groups_used]
H, p_kw = (kruskal(*vals) if len(vals) >= 2 else (np.nan, np.nan))

kw_summary = pd.DataFrame([{
    "test": "Kruskal–Wallis (omnibus)",
    "groups_used": ", ".join(groups_used),
    "H": float(H) if np.isfinite(H) else np.nan,
    "p": float(p_kw) if np.isfinite(p_kw) else np.nan,
}])

# Dunn post-hoc (Bonferroni adjustment handled inside posthoc_dunn)
posthoc_df = pd.DataFrame(columns=["comparison", "n_a", "n_b", "median_a", "median_b", "p_adj"])

if len(groups_used) >= 2:
    df_ph = df.loc[df[group_col].astype(str).isin(groups_used), [group_col, metric]].dropna().copy()
    df_ph[group_col] = df_ph[group_col].astype(str)

    pmat = sp_post.posthoc_dunn(
        df_ph,
        val_col=metric,
        group_col=group_col,
        p_adjust="bonferroni",
    )

    # only compute pairs among groups_used, but keep ORDER-based ordering
    pairs = [(order[i], order[j]) for i in range(len(order)) for j in range(i + 1, len(order))]
    pairs = [(a, b) for (a, b) in pairs if (a in groups_used and b in groups_used)]

    rows = []
    for a, b in pairs:
        x = df_ph.loc[df_ph[group_col] == a, metric].to_numpy()
        y = df_ph.loc[df_ph[group_col] == b, metric].to_numpy()
        rows.append({
            "comparison": f"{a} vs {b}",
            "n_a": int(len(x)),
            "n_b": int(len(y)),
            "median_a": float(np.median(x)) if len(x) else np.nan,
            "median_b": float(np.median(y)) if len(y) else np.nan,
            "p_adj": float(pmat.loc[a, b]),  # Bonferroni-adjusted p-value
        })

    posthoc_df = pd.DataFrame(rows)
    posthoc_df["comparison"] = pd.Categorical(
        posthoc_df["comparison"],
        categories=[f"{a} vs {b}" for a, b in pairs],
        ordered=True,
    )
    posthoc_df = posthoc_df.sort_values("comparison")

# display/print (same objects your plot code expects: df, metric, posthoc_df, p_kw)
try:
    display(kw_summary)
    display(posthoc_df)
except NameError:
    print(kw_summary.to_string(index=False))
    if len(posthoc_df):
        print(posthoc_df.to_string(index=False))
    else:
        print("(posthoc_df is empty: need >=2 groups with data)")